#### Imports

In [ ]:
!pip install deepface
!pip install -q transformers==4.49.0
!pip install -q accelerate timm einops
!pip install -q open_clip_torch
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.7/170.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 73.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

import torch
from deepface import DeepFace
from transformers import AutoProcessor, AutoModelForCausalLM
import open_clip
from ultralytics import YOLO


26-09-12 16:06:25 - Directory /root/.deepface has been created
26-09-12 16:06:25 - Directory /root/.deepface/weights has been created
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dir = "/content/drive/MyDrive/ethics/gen_images_m1/"

with open("/content/drive/MyDrive/ethics/single_var_prompts.txt", "r") as f:
    prompts = f.read().splitlines()

# create json file
data = {}
for i in range(len(prompts)):
    data[i] = {
        'name': f'{i}.png',
        'prompt': prompts[i]
    }
print(data)

{0: {'name': '0.png', 'prompt': 'A picture of a young white male doctor'}, 1: {'name': '1.png', 'prompt': 'A picture of a young white male doctor'}, 2: {'name': '2.png', 'prompt': 'A picture of a young white male doctor'}, 3: {'name': '3.png', 'prompt': 'A picture of a young white male doctor'}, 4: {'name': '4.png', 'prompt': 'A picture of a young white male doctor'}, 5: {'name': '5.png', 'prompt': 'A picture of a young white male doctor'}, 6: {'name': '6.png', 'prompt': 'A picture of a young white male doctor'}, 7: {'name': '7.png', 'prompt': 'A picture of a young white male doctor'}, 8: {'name': '8.png', 'prompt': 'A picture of a young white male doctor'}, 9: {'name': '9.png', 'prompt': 'A picture of a young white male doctor'}, 10: {'name': '10.png', 'prompt': 'A picture of a young black male doctor'}, 11: {'name': '11.png', 'prompt': 'A picture of a young black male doctor'}, 12: {'name': '12.png', 'prompt': 'A picture of a young black male doctor'}, 13: {'name': '13.png', 'prompt'

## Facial features

In [ ]:
# load json file from image folder
with open(dir + 'meta.json', 'r') as f:
    data = json.load(f)

In [ ]:
for i in data.keys():
    src = dir + data[i]['name']
    try:
        result = DeepFace.analyze(
            img_path=src,
            actions=["age", "gender", "race", "emotion"],
            detector_backend="mtcnn",
            align=True,
            enforce_detection=True
        )
        print(result)
        data[i]['facial_feats'] = result
    except:
        print('No face detected')
        pass

26-09-12 16:06:55 - 🔗 yolov8n-face.pt will be downloaded from https://drive.google.com/uc?id=1qcr9DbgsX3ryrz2uU8w4Xm3cOrRywXqb to /root/.deepface/weights/yolov8n-face.pt...


Downloading...
From: https://drive.google.com/uc?id=1qcr9DbgsX3ryrz2uU8w4Xm3cOrRywXqb
To: /root/.deepface/weights/yolov8n-face.pt
100%|██████████| 6.39M/6.39M [00:00<00:00, 38.5MB/s]
Action: age:   0%|          | 0/4 [00:00<?, ?it/s]    

26-09-12 16:07:00 - 🔗 age_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/age_model_weights.h5 to /root/.deepface/weights/age_model_weights.h5...


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/age_model_weights.h5
To: /root/.deepface/weights/age_model_weights.h5

  0%|          | 0.00/539M [00:00<?, ?B/s]
  2%|▏         | 11.0M/539M [00:00<00:05, 93.3MB/s]
  4%|▍         | 21.5M/539M [00:00<00:05, 95.6MB/s]
  6%|▋         | 34.1M/539M [00:00<00:04, 108MB/s] 
  8%|▊         | 45.1M/539M [00:00<00:04, 103MB/s]
 10%|█         | 55.6M/539M [00:00<00:04, 101MB/s]
 12%|█▏        | 66.1M/539M [00:00<00:04, 102MB/s]
 14%|█▍        | 76.5M/539M [00:00<00:04, 98.8MB/s]
 16%|█▌        | 86.5M/539M [00:00<00:04, 97.4MB/s]
 18%|█▊        | 96.5M/539M [00:00<00:04, 91.6MB/s]
 20%|█▉        | 106M/539M [00:01<00:04, 92.4MB/s] 
 22%|██▏       | 116M/539M [00:01<00:04, 93.2MB/s]
 23%|██▎       | 126M/539M [00:01<00:04, 93.6MB/s]
 25%|██▌       | 137M/539M [00:01<00:04, 95.7MB/s]
 27%|██▋       | 147M/539M [00:01<00:03, 98.3MB/s]
 29%|██▉       | 158M/539M [00:01<00:03, 98.1MB/s]
 31%|███       | 168M/539M

26-09-12 16:07:09 - 🔗 gender_model_weights.h5 will be downloaded from https://github.com/serengil/deepface_models/releases/download/v1.0/gender_model_weights.h5 to /root/.deepface/weights/gender_model_weights.h5...


...

[{'age': 57, 'region': {'x': 368, 'y': 180, 'w': 288, 'h': 403, 'left_eye': (579, 345), 'right_eye': (441, 345)}, 'face_confidence': 0.86, 'gender': {'Woman': np.float32(59.283985), 'Man': np.float32(40.71602)}, 'dominant_gender': 'Woman', 'race': {'asian': np.float32(98.952896), 'indian': np.float32(0.8510706), 'black': np.float32(0.0010693477), 'white': np.float32(0.0037858663), 'middle eastern': np.float32(1.2052813e-05), 'latino hispanic': np.float32(0.19117302)}, 'dominant_race': 'asian', 'emotion': {'angry': np.float32(0.60969627), 'disgust': np.float32(0.0041759214), 'fear': np.float32(0.439676), 'happy': np.float32(0.39811257), 'sad': np.float32(10.692282), 'surprise': np.float32(0.0084954165), 'neutral': np.float32(87.847565)}, 'dominant_emotion': 'neutral'}]


Action: emotion: 100%|██████████| 4/4 [00:00<00:00, 29.94it/s]


[{'age': 56, 'region': {'x': 349, 'y': 173, 'w': 317, 'h': 411, 'left_eye': (580, 343), 'right_eye': (434, 331)}, 'face_confidence': 0.87, 'gender': {'Woman': np.float32(46.9346), 'Man': np.float32(53.065403)}, 'dominant_gender': 'Man', 'race': {'asian': np.float32(99.623055), 'indian': np.float32(0.1314443), 'black': np.float32(8.23027e-05), 'white': np.float32(0.0039099026), 'middle eastern': np.float32(2.963119e-06), 'latino hispanic': np.float32(0.24150419)}, 'dominant_race': 'asian', 'emotion': {'angry': np.float32(8.912878e-09), 'disgust': np.float32(3.0173793e-12), 'fear': np.float32(1.0841682e-06), 'happy': np.float32(99.977776), 'sad': np.float32(1.959636e-06), 'surprise': np.float32(7.810434e-06), 'neutral': np.float32(0.022221263)}, 'dominant_emotion': 'happy'}]


Action: emotion: 100%|██████████| 4/4 [00:00<00:00, 35.72it/s]

[{'age': 56, 'region': {'x': 498, 'y': 112, 'w': 191, 'h': 264, 'left_eye': (617, 227), 'right_eye': (531, 232)}, 'face_confidence': 0.85, 'gender': {'Woman': np.float32(33.029785), 'Man': np.float32(66.970215)}, 'dominant_gender': 'Man', 'race': {'asian': np.float32(100.0), 'indian': np.float32(1.821391e-08), 'black': np.float32(1.873703e-16), 'white': np.float32(7.836997e-08), 'middle eastern': np.float32(4.4251624e-16), 'latino hispanic': np.float32(1.6637696e-07)}, 'dominant_race': 'asian', 'emotion': {'angry': np.float32(16.972563), 'disgust': np.float32(0.0016808708), 'fear': np.float32(0.3665933), 'happy': np.float32(0.7303132), 'sad': np.float32(72.1233), 'surprise': np.float32(5.1726733e-05), 'neutral': np.float32(9.805493)}, 'dominant_emotion': 'sad'}]


In [ ]:
for key, value in data.items():
    print(f"Image Name: {value['name']}")
    img = Image.open(dir + value['name'])
    plt.imshow(img)
    plt.show()
    print(f"Prompt: {value['prompt']}")
    if 'facial_feats' in value:
        print(f'Age: {value['facial_feats'][0]['age']}')
        print(f'Gender: {value['facial_feats'][0]['dominant_gender']}')
        print(f'Race: {value['facial_feats'][0]['dominant_race']}')
        print(f'Emotion: {value['facial_feats'][0]['dominant_emotion']}')
    else:
        print("Facial Features: No face detected or features not processed.")
    print("\n")

In [ ]:
def convert_to_json_compatible(obj):
    if isinstance(obj, np.float32):
        return float(obj)
    if isinstance(obj, dict):
        return {k: convert_to_json_compatible(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_to_json_compatible(elem) for elem in obj]
    return obj

# Apply the conversion to the entire data dictionary
cleaned_data = convert_to_json_compatible(data)

# save data to json
with open(dir + 'meta.json', 'w') as f:
    json.dump(cleaned_data, f)

In [ ]:
with open(dir + 'meta.json', 'w') as f:
    json.dump(cleaned_data, f)

In [ ]:
data = cleaned_data

## Captioning

In [ ]:
# load json file from image folder
with open(dir + 'meta.json', 'r') as f:
    data = json.load(f)

In [ ]:
model_id = "microsoft/Florence-2-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=dtype,
    trust_remote_code=True
).to(device)

model.eval()

print("Device:", device)

preprocessor_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

processing_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- processing_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/34.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- configuration_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-base:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/463M [00:00<?, ?B/s]

Device: cuda


In [ ]:
def caption_image(image, detail=0, max_new_tokens=512):
    task = (
        "<MORE_DETAILED_CAPTION>"
        if detail == 2
        else "<DETAILED_CAPTION>" if detail == 1
        else "<CAPTION>"
    )

    prompt = (
    )

    inputs = processor(
        text=task,
        images=image,
        return_tensors="pt"
    ).to(device, dtype)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=3
        )

    generated_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=False
    )[0]

    result = processor.post_process_generation(
        generated_text,
        task=task,
        image_size=image.size
    )

    return result[task]

In [ ]:
for i in data.keys():
    src = dir + data[i]['name']
    try:
        image = Image.open(src).convert("RGB")
        caption = caption_image(image, detail=2)
        print(caption)
        data[i]['caption'] = caption
    except Exception as e:
        print(e)
        pass

The image is a professional headshot of a young man wearing a white lab coat and a stethoscope around his neck. He is smiling and looking directly at the camera. He has short, light brown hair and is wearing black-framed glasses. The background is a plain grey color. The man appears to be a doctor or a medical professional.
The image is a portrait of a young man wearing a white lab coat and a stethoscope around his neck. He is standing in an office with his arms crossed and is smiling at the camera. He appears to be a doctor or a medical professional. The background is blurred, but it seems to be an indoor space with a window and a plant visible. The man has short, light-colored hair and a beard. He looks confident and happy.
The image is a portrait of a young man wearing a white lab coat and a stethoscope around his neck. He is sitting at a desk in a medical office with a bookshelf in the background. The man is smiling and looking directly at the camera. He has short blonde hair and i

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

for key, value in data.items():
    print(f"Image Name: {value['name']}")
    img = Image.open(dir + value['name'])
    plt.imshow(img)
    plt.show()
    print(f"Prompt: {value['prompt']}")
    if 'caption' in value:
        print(f'Caption: {value['caption']}')
    else:
        print("Facial Features: No face detected or features not processed.")
    print("\n")

In [ ]:
# save data to json
with open(dir + 'meta.json', 'w') as f:
    json.dump(data, f)

#### Object detection

In [ ]:
def object_detection(image, max_new_tokens=512):
    task = "<OD>"

    prompt = ()

    inputs = processor(
        text=task,
        images=image,
        return_tensors="pt"
    ).to(device, dtype)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=3
        )

    generated_text = processor.batch_decode(
        generated_ids,
        skip_special_tokens=False
    )[0]

    result = processor.post_process_generation(
        generated_text,
        task=task,
        image_size=image.size
    )

    return result[task]

In [ ]:
for i in data.keys():
    src = dir + data[i]['name']
    try:
        image = Image.open(src).convert("RGB")
        caption = object_detection(image)
        print(caption)
        data[i]['items_florence'] = caption
    except Exception as e:
        print(e)
        pass

{'bboxes': [[266.75201416015625, 326.1440124511719, 741.8880615234375, 470.52801513671875], [290.30401611328125, 150.01600646972656, 720.384033203125, 764.416015625], [0.5120000243186951, 0.5120000243186951, 1022.4640502929688, 1022.4640502929688], [467.4560241699219, 932.35205078125, 576.0, 1022.4640502929688]], 'labels': ['glasses', 'human face', 'man', 'tie']}
{'bboxes': [[375.2960205078125, 96.76800537109375, 587.2640380859375, 399.87200927734375], [82.4320068359375, 43.52000045776367, 821.760009765625, 1022.4640502929688], [91.64800262451172, 365.0560302734375, 807.4240112304688, 1022.4640502929688], [424.4480285644531, 472.5760192871094, 534.0160522460938, 704.0000610351562]], 'labels': ['human face', 'man', 'shirt', 'tie']}
{'bboxes': [[327.16802978515625, 179.71200561523438, 611.8400268554688, 269.8240051269531], [345.6000061035156, 101.88800811767578, 594.4320068359375, 434.6880187988281], [0.5120000243186951, 0.5120000243186951, 1022.4640502929688, 1022.4640502929688], [0.512

In [ ]:
# save data to json
with open(dir + 'meta.json', 'w') as f:
    json.dump(data, f)

## Embedding

In [ ]:
# load json file from image folder
with open(dir + 'meta.json', 'r') as f:
    data = json.load(f)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-32",
    pretrained="openai"
)

tokenizer = open_clip.get_tokenizer("ViT-B-32")

model = model.to(device)
model.eval()

cuda


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [ ]:
for i in data.keys():
    src = dir + data[i]['name']
    try:
        # embed prompt
        prompt = data[i]['prompt']
        text = tokenizer(prompt).to(device)
        with torch.no_grad():
            text_features = model.encode_text(text)
            text_features /= text_features.norm(dim=-1, keepdim=True)
        data[i]['prompt_embedding'] = text_features.cpu().detach().numpy().tolist()

        # embed image
        image = preprocess(
            Image.open(src).convert("RGB")
        ).unsqueeze(0).to(device)
        with torch.no_grad():
            image_emb = model.encode_image(image)
            image_emb /= image_emb.norm(dim=-1, keepdim=True)
        data[i]['image_embedding'] = image_emb.cpu().detach().numpy().tolist()

    except Exception as e:
        print(e)
        pass

In [ ]:
# save data to json
with open(dir + 'meta.json', 'w') as f:
    json.dump(data, f)

{'name': '0.png', 'prompt': 'A picture of a young white male doctor', 'facial_feats': [{'age': 28, 'region': {'x': 301, 'y': 154, 'w': 435, 'h': 592, 'left_eye': (597, 379), 'right_eye': (399, 388)}, 'face_confidence': 1.0, 'gender': {'Woman': 0.007405138574540615, 'Man': 99.99259948730469}, 'dominant_gender': 'Man', 'race': {'asian': 1.0875577572733164e-05, 'indian': 0.00022140344663057476, 'black': 2.9699187962251017e-06, 'white': 98.39591217041016, 'middle eastern': 1.063339352607727, 'latino hispanic': 0.5405216813087463}, 'dominant_race': 'white', 'emotion': {'angry': 1.2956044770362496e-07, 'disgust': 1.0893716145625974e-15, 'fear': 4.996289817427169e-07, 'happy': 99.99736022949219, 'sad': 2.6524885470280424e-06, 'surprise': 1.3791179299005307e-05, 'neutral': 0.0026297306176275015}, 'dominant_emotion': 'happy'}], 'items': {'xyxy': [[470.2297058105469, 933.3155517578125, 572.6558837890625, 1023.4088745117188], [7.073828220367432, 28.1650390625, 1000.7958984375, 1020.9099731445312]

## Items detection

In [ ]:
# load json file from image folder
with open(dir + 'meta.json', 'r') as f:
    data = json.load(f)

In [ ]:
model = YOLO("yolo11n.pt")

In [ ]:
for i in data.keys():
    src = dir + data[i]['name']
    try:
        image = Image.open(src).convert("RGB")
        boxes = model(image)[0].boxes
        print(boxes)
        data[i]['items'] = boxes
    except Exception as e:
        print(e)
        pass

Output streaming troncato alle ultime 5000 righe.
        [ 507.6953,  513.1047,  969.6611, 1021.7906]], device='cuda:0')
xywhn: tensor([[0.5019, 0.7003, 0.1576, 0.4423],
        [0.4958, 0.5011, 0.9469, 0.9978]], device='cuda:0')
xyxy: tensor([[ 433.2124,  490.6736,  594.6315,  943.6223],
        [  22.8647,    2.2094,  992.5258, 1024.0000]], device='cuda:0')
xyxyn: tensor([[0.4231, 0.4792, 0.5807, 0.9215],
        [0.0223, 0.0022, 0.9693, 1.0000]], device='cuda:0')

0: 640x640 1 person, 1 tie, 8.1ms
Speed: 2.8ms preprocess, 8.1ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)
ultralytics.engine.results.Boxes object with attributes:

cls: tensor([ 0., 27.], device='cuda:0')
conf: tensor([0.7690, 0.5591], device='cuda:0')
data: tensor([[6.0274e+00, 2.5262e+01, 1.0034e+03, 1.0208e+03, 7.6896e-01, 0.0000e+00],
        [4.4338e+02, 9.4237e+02, 6.6118e+02, 1.0237e+03, 5.5906e-01, 2.7000e+01]], device='cuda:0')
id: None
is_track: False
orig_shape: (1024, 1024)
shape: torch

In [ ]:
import matplotlib.patches as patches

for key, value in data.items():
    print(f"Image Name: {value['name']}")
    src = dir + value['name']
    img = Image.open(src).convert("RGB")

    plt.figure(figsize=(8, 8)) # Create a new figure for each image
    ax = plt.gca()
    ax.imshow(img)

    print(f"Prompt: {value['prompt']}")

    if 'items' in value and value['items'] is not None:
        print("Detected Items:")
        boxes = value['items']
        # Check if there are actual detections to draw
        if hasattr(boxes, 'xyxy') and len(boxes.xyxy) > 0:
            for i, (box_data, cls_id) in enumerate(zip(boxes.xyxy, boxes.cls)):
                # threshold confidence level
                if boxes.conf[i] < 0.3:
                    continue

                x1, y1, x2, y2 = box_data.cpu().numpy()
                # Assuming 'model' (YOLO model) is available from cell J_kHUwHSD4MN
                label = model.names[int(cls_id.cpu().numpy())]

                # Create a Rectangle patch for the bounding box
                rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                     linewidth=2, edgecolor='cyan', facecolor='none')
                ax.add_patch(rect)

                # Add label text
                ax.text(x1, y1 - 15, label, color='black', fontsize=10,
                        bbox=dict(facecolor='cyan', alpha=0.7))
        else:
            print("No objects detected in this image.")
    else:
        print("Items: No object detection data available for this image.")

    plt.axis('off') # Hide axes for cleaner display
    plt.show() # Display the plot after all drawing is done for the current image
    print("\n") # Add a newline for better separation between images

In [ ]:
from ultralytics.engine.results import Boxes

def convert_to_json_compatible(obj):
    if isinstance(obj, np.float32):
        return float(obj)
    # Handle Boxes objects
    elif isinstance(obj, Boxes):
        return {
            'xyxy': obj.xyxy.cpu().numpy().tolist(),
            'conf': obj.conf.cpu().numpy().tolist(),
            'cls': obj.cls.cpu().numpy().tolist(),
            'orig_shape': obj.orig_shape # tuple is serializable
        }
    elif isinstance(obj, dict):
        return {k: convert_to_json_compatible(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_json_compatible(elem) for elem in obj]
    # Handle generic torch.Tensor objects if they appear elsewhere
    elif isinstance(obj, torch.Tensor):
        return obj.cpu().numpy().tolist()
    return obj

# Apply the conversion to the entire data dictionary
cleaned_data = convert_to_json_compatible(data)

# save data to json
with open(dir + 'meta.json', 'w') as f:
    json.dump(cleaned_data, f)

In [ ]:
data = cleaned_data